# Black Swan V3 - Date (Binary Classification)
**Ce e nou fata de V2:**
- Target binar: 1 = V-shape, 0 = Non-V (L + U combinate)
- Tot restul identic cu V2

## 1. Import Librarii

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('plots', exist_ok=True)

print('Librarii incarcate.')

Librarii incarcate.


## 2. Definitia Evenimentelor (identic V2)

In [2]:
EVENTS = [
    {'name': 'Dotcom Crash',           'start': '2000-01-01', 'end': '2002-10-31', 'category': 'Grey Swan'},
    {'name': 'Global Financial Crisis','start': '2008-01-01', 'end': '2009-06-30', 'category': 'Grey Swan'},
    {'name': 'European Debt Crisis',   'start': '2011-04-01', 'end': '2011-12-31', 'category': 'Grey Swan'},
    {'name': 'Taper Tantrum',          'start': '2013-04-01', 'end': '2013-10-31', 'category': 'Grey Swan'},
    {'name': 'China Devaluation',      'start': '2015-06-01', 'end': '2016-02-28', 'category': 'Grey Swan'},
    {'name': 'Fed Rate Hikes 2022',    'start': '2022-01-01', 'end': '2022-12-31', 'category': 'Grey Swan'},
    {'name': 'Liberation Day Tariffs', 'start': '2025-02-01', 'end': '2025-08-01', 'category': 'Grey Swan'},
    {'name': 'COVID Crash',            'start': '2020-01-15', 'end': '2020-08-31', 'category': 'Black Swan'},
    {'name': '9/11',                   'start': '2001-07-01', 'end': '2001-12-31', 'category': 'Black Swan'},
    {'name': 'Flash Crash',            'start': '2010-04-01', 'end': '2010-08-31', 'category': 'Black Swan'},
]

print('Evenimente:', len(EVENTS))

Evenimente: 10


## 3. Descarcare Date

In [3]:
sp500_raw = yf.download('^GSPC', start='1999-01-01', end='2025-09-01', auto_adjust=True)
sp500_raw = sp500_raw[['Close', 'Volume']].copy()
sp500_raw.columns = ['SP500_Close', 'SP500_Volume']

vix_raw = yf.download('^VIX', start='1999-01-01', end='2025-09-01', auto_adjust=True)
vix_raw = vix_raw[['Close']].copy()
vix_raw.columns = ['VIX']

market_data = sp500_raw.join(vix_raw, how='left')
market_data.index = pd.to_datetime(market_data.index)

print('Date descarcate:', market_data.index[0].date(), '->', market_data.index[-1].date())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Date descarcate: 1999-01-04 -> 2025-08-29


## 4. Feature Engineering (identic V2)

In [4]:
df = market_data.copy()

# Features V1
df['Return_1d']        = df['SP500_Close'].pct_change()
df['Return_5d']        = df['SP500_Close'].pct_change(5)
df['MA50']             = df['SP500_Close'].rolling(50).mean()
df['MA200']            = df['SP500_Close'].rolling(200).mean()
df['Dist_MA50']        = (df['SP500_Close'] - df['MA50'])  / df['MA50']
df['Dist_MA200']       = (df['SP500_Close'] - df['MA200']) / df['MA200']
df['Dist_52w_High']    = (df['SP500_Close'] - df['SP500_Close'].rolling(252).max()) / df['SP500_Close'].rolling(252).max()
df['VIX_Change']       = df['VIX'].pct_change()
df['Realized_Vol_10d'] = df['Return_1d'].rolling(10).std() * np.sqrt(252)
df['Volume_Ratio']     = df['SP500_Volume'] / df['SP500_Volume'].rolling(20).mean()
df['Consec_Down']      = (
    df['Return_1d'].lt(0)
    .groupby((df['Return_1d'].lt(0) != df['Return_1d'].lt(0).shift()).cumsum())
    .cumsum()
)

def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))

df['RSI'] = compute_rsi(df['SP500_Close'])

# Features V2
def rolling_slope(series, window):
    slopes = [np.nan] * len(series)
    vals   = series.values
    x      = np.arange(window)
    for i in range(window - 1, len(vals)):
        y = vals[i - window + 1 : i + 1]
        if not np.any(np.isnan(y)):
            slopes[i] = np.polyfit(x, y, 1)[0]
    return pd.Series(slopes, index=series.index)

df['VIX_Trend_20d']   = rolling_slope(df['VIX'], 20)
df['SP500_Trend_20d'] = rolling_slope(df['SP500_Close'], 20)
df['Local_Min_20d']   = df['SP500_Close'].rolling(20).min()
df['Dist_Local_Min']  = (df['SP500_Close'] - df['Local_Min_20d']) / df['Local_Min_20d']
df['VIX_MA60']        = df['VIX'].rolling(60).mean()
df['VIX_Ratio']       = df['VIX'] / df['VIX_MA60']

print('Features calculate.')

Features calculate.


## 5. Label Binar + Constructie Dataset

**Singura diferenta fata de V2:**
- `1` = V-shape (forward return > 5% in 60 zile)
- `0` = Non-V (L-shape sau U-shape, orice altceva)

Nu mai avem 3 clase, ci una singura intrebare: **piata va recupera rapid sau nu?**

In [5]:
V_THRESHOLD  = 0.05
FORWARD_DAYS = 60

df['Forward_Return_60d'] = df['SP500_Close'].shift(-FORWARD_DAYS) / df['SP500_Close'] - 1

# Label binar: 1 = V-shape, 0 = Non-V
df['label'] = (df['Forward_Return_60d'] > V_THRESHOLD).astype(int)

feature_cols = [
    'Return_1d', 'Return_5d', 'Dist_MA50', 'Dist_MA200',
    'Dist_52w_High', 'VIX', 'VIX_Change', 'Realized_Vol_10d',
    'Volume_Ratio', 'Consec_Down', 'RSI',
    'VIX_Trend_20d', 'SP500_Trend_20d', 'Dist_Local_Min', 'VIX_Ratio', 'Phase'
]

all_windows = []

for event in EVENTS:
    mask   = (df.index >= event['start']) & (df.index <= event['end'])
    window = df.loc[mask].copy()

    if len(window) == 0:
        continue

    n = len(window)
    phases = [1 if i <= 30 else (2 if i <= 90 else 3) for i in range(n)]
    window['Phase']        = phases
    window['event_name']   = event['name']
    window['category']     = event['category']
    window['day_relative'] = range(n)

    all_windows.append(window)

dataset = pd.concat(all_windows)
dataset = dataset.dropna(subset=feature_cols + ['label'])

print('Dataset final:', len(dataset), 'randuri')
print('\nDistributia labelurilor:')
counts = dataset['label'].value_counts()
print('  Non-V (0):', counts[0], '(' + str(round(counts[0]/len(dataset)*100, 1)) + '%)')
print('  V-shape (1):', counts[1], '(' + str(round(counts[1]/len(dataset)*100, 1)) + '%)')
print('\nDistributie per eveniment:')
pivot = dataset.groupby(['event_name', 'label']).size().unstack(fill_value=0)
pivot.columns = ['Non-V (0)', 'V-shape (1)']
pivot['% V-shape'] = (pivot['V-shape (1)'] / (pivot['Non-V (0)'] + pivot['V-shape (1)']) * 100).round(1)
print(pivot)

Dataset final: 2381 randuri

Distributia labelurilor:
  Non-V (0): 1697 (71.3%)
  V-shape (1): 684 (28.7%)

Distributie per eveniment:
                         Non-V (0)  V-shape (1)  % V-shape
event_name                                                
9/11                            95           28       22.8
COVID Crash                     55          104       65.4
China Devaluation              132           56       29.8
Dotcom Crash                   611          100       14.1
European Debt Crisis           109           81       42.6
Fed Rate Hikes 2022            217           34       13.5
Flash Crash                     59           47       44.3
Global Financial Crisis        268          109       28.9
Liberation Day Tariffs          66           59       47.2
Taper Tantrum                   85           66       43.7


## 6. Train / Test Split

In [6]:
TRAIN_EVENTS = [
    'Global Financial Crisis', 'European Debt Crisis',
    'Taper Tantrum', 'China Devaluation', '9/11', 'Flash Crash'
]
TEST_EVENTS = [
    'Fed Rate Hikes 2022', 'COVID Crash', 'Liberation Day Tariffs'
]

dotcom_all   = dataset[dataset['event_name'] == 'Dotcom Crash']
dotcom_train = dotcom_all.iloc[:-200]
dotcom_test  = dotcom_all.iloc[-200:]

train_df = pd.concat([dotcom_train, dataset[dataset['event_name'].isin(TRAIN_EVENTS)]])
test_df  = pd.concat([dotcom_test,  dataset[dataset['event_name'].isin(TEST_EVENTS)]])

train_df = train_df.dropna(subset=feature_cols)
test_df  = test_df.dropna(subset=feature_cols)

print('Train:', len(train_df), 'zile')
print('Test: ', len(test_df),  'zile')

for name, split in [('Train', train_df), ('Test', test_df)]:
    c = split['label'].value_counts()
    print('\nDistributie', name + ':')
    print('  Non-V (0):', c.get(0, 0))
    print('  V-shape (1):', c.get(1, 0))

Train: 1646 zile
Test:  735 zile

Distributie Train:
  Non-V (0): 1185
  V-shape (1): 461

Distributie Test:
  Non-V (0): 512
  V-shape (1): 223


## 7. Salvare Dataset

In [7]:
dataset.to_csv('v3_dataset.csv')
train_df.to_csv('v3_train.csv')
test_df.to_csv('v3_test.csv')

with open('v3_feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

print('Fisiere salvate:')
print('  v3_dataset.csv ->', len(dataset), 'randuri')
print('  v3_train.csv   ->', len(train_df), 'randuri')
print('  v3_test.csv    ->', len(test_df), 'randuri')
print('  v3_feature_cols.json')
print('\nGata pentru v3_modeling.ipynb')

Fisiere salvate:
  v3_dataset.csv -> 2381 randuri
  v3_train.csv   -> 1646 randuri
  v3_test.csv    -> 735 randuri
  v3_feature_cols.json

Gata pentru v3_modeling.ipynb
